# 29 — Interpretability

**Context Demystifies Forecasting**

The earlier notebooks showed that latent constraints can improve forecasting.

This notebook asks a different question:

> What does the latent state represent?

We visualize:

1. physical latent coordinates
2. residual latent behavior
3. constrained vs unconstrained trajectories
4. where forecast error appears in latent space

The goal is to turn forecasting performance into an interpretable state-space story.


## Core idea

A model can be accurate without being interpretable.

For Phys-JEPA-style forecasting, the stronger claim is:

```text
latent_state = physical_state + residual_state
```

and the physical component should align with measurable structure.

In this notebook, the physical latent state is represented by seasonal coordinates:

```text
sin(2πt / period), cos(2πt / period)
```

That gives us a phase-space view of the constrained state.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIGURES = ROOT / "figures"
RESULTS = ROOT / "results"

FIGURES.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)

rng = np.random.default_rng(260616076)

print(f"ROOT: {ROOT}")
print(f"FIGURES: {FIGURES}")
print(f"RESULTS: {RESULTS}")

## Build a climate-style signal

We reuse a climate-style temperature signal:

```text
temperature = seasonal structure + residual variation
```

The point is not climate realism. The point is interpretability: can we see the structured state separately from the residual state?


In [ ]:
days = np.arange(0, 1460)

period = 365

seasonal_latent_sin = np.sin(2 * np.pi * days / period)
seasonal_latent_cos = np.cos(2 * np.pi * days / period)

physical_temperature = 15 + 10 * seasonal_latent_sin
residual_temperature = (
    0.8 * rng.normal(size=len(days))
    + 0.8 * np.sin(2 * np.pi * days / 37 + 0.5)
)

temperature = physical_temperature + residual_temperature

df = pd.DataFrame({
    "day": days,
    "latent_sin": seasonal_latent_sin,
    "latent_cos": seasonal_latent_cos,
    "physical_temperature": physical_temperature,
    "residual_temperature": residual_temperature,
    "temperature": temperature,
})

df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(df["day"], df["temperature"], label="observed temperature")
ax.plot(df["day"], df["physical_temperature"], label="physical temperature")

ax.set_title("Climate-style signal and physical component")
ax.set_xlabel("day")
ax.set_ylabel("temperature")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "29_signal_and_physical_component.png", dpi=160)
plt.show()

## Physical latent phase space

The physical latent state forms a closed seasonal loop.

This is an interpretable latent state: position on the loop corresponds to time of year.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

ax.plot(df["latent_cos"], df["latent_sin"])

ax.set_title("Physical latent phase space")
ax.set_xlabel("cos seasonal coordinate")
ax.set_ylabel("sin seasonal coordinate")
ax.set_aspect("equal", adjustable="box")

fig.tight_layout()
fig.savefig(FIGURES / "29_physical_latent_phase_space.png", dpi=160)
plt.show()

## Residual latent behavior

The residual state should not dominate the physical state.

Here we visualize residual temperature against seasonal phase.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.scatter(df["latent_sin"], df["residual_temperature"], s=8, alpha=0.45)

ax.set_title("Residual state against physical phase")
ax.set_xlabel("seasonal latent coordinate: sin")
ax.set_ylabel("residual temperature")

fig.tight_layout()
fig.savefig(FIGURES / "29_residual_vs_physical_phase.png", dpi=160)
plt.show()

## Estimate latent state from observations

We now estimate the seasonal physical component from observed temperature.

This is the same interpretability question from Notebook 13:

> does the estimated physical state align with the true physical state?


In [ ]:
split = 1100

train = df.iloc[:split].copy()
test = df.iloc[split:].copy()

x_train = train["day"].to_numpy()
y_train = train["temperature"].to_numpy()

x_test = test["day"].to_numpy()
y_test = test["temperature"].to_numpy()

def design_matrix(x):
    return np.column_stack([
        np.ones_like(x),
        np.sin(2 * np.pi * x / period),
        np.cos(2 * np.pi * x / period),
    ])

coef, *_ = np.linalg.lstsq(design_matrix(x_train), y_train, rcond=None)

train_physical_est = design_matrix(x_train) @ coef
test_physical_est = design_matrix(x_test) @ coef

train_residual_est = y_train - train_physical_est
test_residual_est = y_test - test_physical_est

coef

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(x_test, y_test, label="truth")
ax.plot(x_test, test_physical_est, label="estimated physical state")
ax.plot(x_test, test_residual_est, label="estimated residual state")

ax.set_title("Estimated physical and residual states")
ax.set_xlabel("day")
ax.set_ylabel("temperature")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "29_estimated_physical_residual_states.png", dpi=160)
plt.show()

## Constrained vs unconstrained latent trajectories

A constrained seasonal latent state remains on the unit circle.

An unconstrained latent state can drift.

We create a simple drifting latent trajectory to visualize the difference.


In [ ]:
test_days = test["day"].to_numpy()

constrained_x = np.cos(2 * np.pi * test_days / period)
constrained_y = np.sin(2 * np.pi * test_days / period)

drift = np.linspace(0, 0.55, len(test_days))
unconstrained_x = (1 + drift) * constrained_x
unconstrained_y = (1 + drift) * constrained_y

fig, ax = plt.subplots(figsize=(6, 6))

ax.plot(constrained_x, constrained_y, label="constrained latent trajectory")
ax.plot(unconstrained_x, unconstrained_y, label="unconstrained drifting trajectory")

ax.set_title("Constrained vs unconstrained latent state")
ax.set_xlabel("latent coordinate 1")
ax.set_ylabel("latent coordinate 2")
ax.set_aspect("equal", adjustable="box")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "29_constrained_vs_unconstrained_latent.png", dpi=160)
plt.show()

## Decode trajectories into forecasts

The constrained trajectory preserves seasonal amplitude.

The unconstrained trajectory drifts away from the learned physical state.


In [ ]:
intercept, sin_coef, cos_coef = coef

constrained_forecast = (
    intercept
    + sin_coef * constrained_y
    + cos_coef * constrained_x
)

unconstrained_forecast = (
    intercept
    + sin_coef * unconstrained_y
    + cos_coef * unconstrained_x
)

forecast_df = pd.DataFrame({
    "day": test_days,
    "truth": y_test,
    "constrained_forecast": constrained_forecast,
    "unconstrained_forecast": unconstrained_forecast,
})

forecast_df.to_csv(RESULTS / "29_interpretability_forecasts.csv", index=False)

forecast_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

ax.plot(forecast_df["day"], forecast_df["truth"], label="truth")
ax.plot(forecast_df["day"], forecast_df["constrained_forecast"], label="constrained latent forecast")
ax.plot(forecast_df["day"], forecast_df["unconstrained_forecast"], label="unconstrained latent forecast")

ax.set_title("Forecasts decoded from latent trajectories")
ax.set_xlabel("day")
ax.set_ylabel("temperature")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "29_decoded_latent_forecasts.png", dpi=160)
plt.show()

## Error in latent space

If errors grow because latent trajectories drift, then forecast error should correlate with distance from the constrained physical manifold.

For the seasonal example, the physical manifold is the unit circle.


In [ ]:
def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

forecast_df["constrained_error"] = forecast_df["truth"] - forecast_df["constrained_forecast"]
forecast_df["unconstrained_error"] = forecast_df["truth"] - forecast_df["unconstrained_forecast"]

forecast_df["unconstrained_radius"] = np.sqrt(unconstrained_x**2 + unconstrained_y**2)
forecast_df["distance_from_physical_manifold"] = np.abs(forecast_df["unconstrained_radius"] - 1.0)

metrics = pd.DataFrame([
    {
        "method": "constrained_latent",
        "RMSE": rmse(forecast_df["truth"], forecast_df["constrained_forecast"]),
    },
    {
        "method": "unconstrained_latent",
        "RMSE": rmse(forecast_df["truth"], forecast_df["unconstrained_forecast"]),
    },
])

metrics.to_csv(RESULTS / "29_interpretability_metrics.csv", index=False)

metrics

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.scatter(
    forecast_df["distance_from_physical_manifold"],
    np.abs(forecast_df["unconstrained_error"]),
    s=10,
    alpha=0.6,
)

ax.set_title("Forecast error vs latent drift")
ax.set_xlabel("distance from physical manifold")
ax.set_ylabel("absolute forecast error")

fig.tight_layout()
fig.savefig(FIGURES / "29_error_vs_latent_drift.png", dpi=160)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.bar(metrics["method"], metrics["RMSE"])
ax.set_title("Latent interpretability RMSE")
ax.set_ylabel("RMSE")

fig.tight_layout()
fig.savefig(FIGURES / "29_interpretability_rmse.png", dpi=160)
plt.show()

## Interpretation

The interpretability result is visual:

```text
constrained physical latent state
    stays on the seasonal manifold

unconstrained latent state
    drifts away from that manifold
```

When the latent state drifts away from measurable structure, decoded forecasts drift too.

This is the repo's interpretability version of the Phys-JEPA claim:

> constraints should shape latent state transitions, not only output predictions.


## Next

Notebook 37 should focus on forecast stability:

```text
How does error accumulate across horizon when latent state is constrained, weakly constrained, or unconstrained?
```

That will connect the interpretability story back to long-horizon forecasting performance.
